# All-in-One Evaluation Notebook for MILP Approaches

## Purpose

This notebook is the central place for documenting, running, and comparing different optimization approaches for the district heating problem.

The goal of this notebook is **evaluation**, not detailed development. Each new approach should be added here in a way that makes later comparison easy, fair, and readable.


## Problem Statement

We aim to minimize the operational cost of the district heating system while meeting heat demand and respecting all technical and market-related constraints.

The system consists of a heat pump, a condensing boiler, a CHP unit, and a thermal storage, while participating in the day-ahead electricity market.

The full mathematical formulation should be treated as the source of truth and referenced here:

- `docs/optimization/optimization_problem.md`


## Fundamental Notebook Rules

1. **This notebook is for comparison, not for messy exploration.**
2. **One code cell = one whole approach.**
3. Every approach must be understandable on its own, except for the shared setup defined above it.
4. No approach may change the input data.
5. Results must be recorded in a consistent format so that approaches can be compared later.
6. If a solver fails, times out, or returns an infeasible solution, that must be documented and not hidden.
7. The notebook should stay readable. Add approaches, do not rewrite previous ones without noting why.


## Practical Guidelines for Adding a New Approach

For each new approach, add exactly two cells:

- one **markdown cell** describing the approach
- one **code cell** containing the complete implementation of that approach

The markdown cell directly above an approach should always answer these questions:

- What is the name of the approach?
- What is the main idea?
- What is changed compared with the canonical problem formulation?
- How should it be interpreted against the shared evaluation baseline?
- What result or behavior do we expect?

The code cell directly below it should contain the full approach, including model setup, solve step, and result collection for that approach.


## Shared Setup

This section is reserved for everything that must be shared by all approaches.

It should contain only notebook-wide setup and no approach-specific logic.

Extract the shared setup from the canonical problem formulation and define it here once for the whole notebook.

Put here:

- imports
- time structure & index definition
- shared data loading
- shared inputs from the problem definition
- fixed parameters and common assumptions
- shared initial conditions
- shared rolling-horizon assumptions
- shared grid assumptions
- shared benchmark definitions
- shared helper functions
- common result storage
- shared plotting helpers

This section should make sure that all later approach cells work on the same common basis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120


def find_project_root(start: Path | None = None) -> Path:
    """Resolve the repository root so notebook paths work from different launch locations."""
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing pyproject.toml and data/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DEMAND_FILE = DATA_DIR / "RawData_MeasuredHeadDemand.csv"
PRICE_FILE = DATA_DIR / "Gro_handelspreise_202403010000_202603020000_Stunde.csv"

# Common time structure
DT_HOURS = 0.25
HOURS_PER_DAY = 24
INTERVALS_PER_HOUR = 4
T = HOURS_PER_DAY * INTERVALS_PER_HOUR
TIME_INDEX = range(1, T + 1)

# Shared inputs and fixed parameters from the canonical problem formulation
PARAMS = {
    "gas_price": 35.0,
    "co2_factor": 0.201,
    "co2_price": 60.0,
    "sto_capacity": 200.0,
    "sto_charge_max": 15.0,
    "sto_discharge_max": 15.0,
    "sto_loss": 0.000125,
    "sto_soc_init": 200.0,
    "hp_p_el_min": 1.0,
    "hp_p_el_max": 8.0,
    "hp_cop": 3.5,
    "boiler_q_min": 2.0,
    "boiler_q_max": 12.0,
    "boiler_eff": 0.97,
    "boiler_min_up": 4,
    "boiler_min_down": 4,
    "chp_p_el_min": 2.0,
    "chp_p_el_max": 6.0,
    "chp_eff_el": 0.40,
    "chp_eff_th": 0.48,
    "chp_min_up": 8,
    "chp_min_down": 8,
    "chp_startup_cost": 600.0,
}

PARAMS["gas_cost"] = PARAMS["gas_price"] + PARAMS["co2_factor"] * PARAMS["co2_price"]
PARAMS["chp_heat_power_ratio"] = PARAMS["chp_eff_th"] / PARAMS["chp_eff_el"]
PARAMS["chp_q_th_min"] = PARAMS["chp_heat_power_ratio"] * PARAMS["chp_p_el_min"]
PARAMS["chp_q_th_max"] = PARAMS["chp_heat_power_ratio"] * PARAMS["chp_p_el_max"]

# Shared initial conditions for the first optimization run
INITIAL_STATE = {
    "hp_on_0": 0,
    "boiler_on_0": 0,
    "chp_on_0": 0,
    "soc_0": PARAMS["sto_soc_init"],
    "boiler_time_in_state_0": 0,
    "chp_time_in_state_0": 0,
}

# Shared rolling-horizon and grid assumptions from the formulation
ROLLING_HORIZON_ASSUMPTIONS = {
    "carry_final_state_between_days": True,
    "first_run_units_off": True,
    "no_terminal_constraint": True,
}

GRID_ASSUMPTIONS = {
    "hp_uses_da_price": True,
    "chp_sells_at_da_price": True,
    "grid_connection_capacity_limit": None,
}

# The exact benchmark set is intentionally left open until the evaluation baseline is fixed.
BENCHMARK_DATES = []

RESULT_COLUMNS = [
    "approach_name",
    "runtime_seconds",
    "notes",
]
RESULTS = pd.DataFrame(columns=RESULT_COLUMNS)


def get_price_column(columns: list[str]) -> str:
    """Select the Germany/Luxembourg DA price column from the SMARD export."""
    for column in columns:
        if column.startswith("Deutschland/Luxemburg [€/MWh]"):
            return column
    raise KeyError("Could not find a Germany/Luxembourg electricity price column.")


def load_heat_demand() -> pd.DataFrame:
    """Load measured heat demand as an hourly UTC-indexed series in MW_th."""
    df = pd.read_csv(DEMAND_FILE)
    df["timestamp"] = pd.to_datetime(df["Time Point"], utc=True)
    df["demand_mw_th"] = df["Measured Heat Demand[W]"] / 1e6
    return df[["timestamp", "demand_mw_th"]].set_index("timestamp").sort_index()


def load_electricity_prices() -> pd.DataFrame:
    """Load day-ahead electricity prices as an hourly UTC-indexed series in EUR/MWh_el."""
    df = pd.read_csv(PRICE_FILE, sep=";", decimal=",", low_memory=False)
    price_column = get_price_column(df.columns.tolist())
    timestamps = pd.to_datetime(df["Datum von"], format="%d.%m.%Y %H:%M")
    timestamps = timestamps.dt.tz_localize("Europe/Berlin", ambiguous="infer", nonexistent="shift_forward")
    df["timestamp"] = timestamps.dt.tz_convert("UTC")
    df["price_eur_mwh"] = df[price_column]
    return df[["timestamp", "price_eur_mwh"]].set_index("timestamp").sort_index()


def expand_hourly_to_quarter_hour(values: pd.Series) -> np.ndarray:
    """Repeat each hourly value across the four 15-minute intervals of the formulation."""
    return np.repeat(values.to_numpy(), INTERVALS_PER_HOUR)


def register_result(result: dict) -> None:
    """Append one standardized result row to the shared result table."""
    missing = [column for column in RESULT_COLUMNS if column not in result]
    if missing:
        raise ValueError(f"Result is missing required fields: {missing}")
    global RESULTS
    RESULTS.loc[len(RESULTS)] = [result[column] for column in RESULT_COLUMNS]


print(f"Project root: {PROJECT_ROOT}")
print(f"Demand file: {DEMAND_FILE.name}")
print(f"Price file: {PRICE_FILE.name}")
print(f"Horizon: {T} intervals at {DT_HOURS} h")
print(f"Effective gas cost: {PARAMS['gas_cost']:.2f} EUR/MWh_Hs")


## Evaluation Agreement

Before approaches are compared, we should agree on a common evaluation baseline outside the individual approach cells.

Once that baseline is fixed, it should be documented here and applied to every approach in the notebook.

This section should later define at least:

- the benchmark instances or days
- the reported evaluation metrics


## Required Output of Each Approach

Each approach should later produce results in a consistent structure so that the final evaluation section can compare them cleanly.

Suggested minimum fields:

- `approach_name`
- `runtime_seconds`
- `notes`

Optional fields can be added later once we agreed on the exact benchmark format.


## Approach Template

Copy the following pattern whenever a new approach is added.

If more approaches are tested later, duplicate one markdown-code pair and append it below the last approach section.

**Markdown cell template:**

- **Approach name:**
- **Main idea:**
- **Difference to canonical problem formulation:**
- **Interpretation against evaluation baseline:**
- **Expectation:**

**Code cell template:**

- build the full model for this approach
- solve it
- collect outputs in the agreed result format
- store or display the results clearly


## Approach 1

- **Approach name:**
- **Main idea:**
- **Difference to canonical problem formulation:**
- **Interpretation against evaluation baseline:**
- **Expectation:**


In [ ]:
# One code cell = one whole approach.
#
# Place the complete implementation of Approach 1 here later.
# The full approach should live in this single cell.


## Approach 2

- **Approach name:**
- **Main idea:**
- **Difference to canonical problem formulation:**
- **Interpretation against evaluation baseline:**
- **Expectation:**


In [ ]:
# One code cell = one whole approach.
#
# Place the complete implementation of Approach 2 here later.
# The full approach should live in this single cell.


## Approach 3

- **Approach name:**
- **Main idea:**
- **Difference to canonical problem formulation:**
- **Interpretation against evaluation baseline:**
- **Expectation:**


In [ ]:
# One code cell = one whole approach.
#
# Place the complete implementation of Approach 3 here later.
# The full approach should live in this single cell.


## Evaluation and Comparison

This section is reserved for the final comparison once multiple approaches have been added.

Later, summarize here:

- which approaches were run
- how each approach performed against the agreed evaluation baseline
- which metrics are compared
- which results are still preliminary


In [ ]:
# Final comparison code goes here later.
#
# This section should aggregate the standardized results of all approaches.
